In [1]:
def sample_narratives_by_product_cluster(df, product, cluster_col, n_samples=10):       
    """
    Sample narratives for a given product and cluster.

    Args:
        df (pd.DataFrame): DataFrame containing complaint data.
        product (str): The product to filter by.
        cluster_col (str): The column name for cluster labels.
        n_samples (int): Number of samples to return per cluster.
        
    Returns:
        list: List of sampled narratives.
    """
    product_df = df[df['product'] == product]
    sampled_narratives = []
    for cluster in product_df[cluster_col].unique():
        cluster_df = product_df[product_df[cluster_col] == cluster]
        sampled_narratives.extend(
            cluster_df['consumer_complaint_narrative'].dropna().sample(
                n=min(n_samples, len(cluster_df)), random_state=42
            ).tolist()
        )
    return sampled_narratives
def create_cluster_name(cluster_id, sample_narratives, model=None):
    """
    Create a descriptive name for a cluster based on sample narratives using huggingface model.

    Args:
        cluster_id (int): The cluster ID.
        sample_narratives (list): List of sample narratives from the cluster.
        model (str): The Hugging Face model to use (default: "openai-community/gpt2").

    Returns:
        str: Descriptive name for the cluster.
    """
    from huggingface_hub import InferenceClient
    
    if model is None:
        model = "Qwen/Qwen2.5-72B-Instruct"
    
    # Combine and truncate narratives to avoid token limits
    combined_text = " ".join(sample_narratives)  # Use first 5 narratives
    # if len(combined_text) > 1000:
    #     combined_text = combined_text[:1000]
    
    # Create prompt for naming the cluster
    user_prompt = f"""Read these customer complaints and respond with ONLY a 2-4 word category name:

{combined_text}

Category:"""
    
    try:
        client = InferenceClient(model=model)
        
        # Use conversational API
        messages = [
            {
                "role": "user",
                "content": user_prompt
            }
        ]
        
        response = client.chat_completion(
            messages=messages,
            max_tokens=20,
            temperature=0.7
        )
        
        # Extract the response text
        print(f"Raw response: {response}")
        message = response.choices[0].message
        # Try content first, then reasoning if content is empty
        cluster_name = message.content if message.content else (message.reasoning if hasattr(message, 'reasoning') else "")
        cluster_name = cluster_name.strip().split('\n')[0]
        # Remove quotes if present
        cluster_name = cluster_name.strip('"\'')
        print(f"Extracted cluster name: {cluster_name}")
        
        return cluster_name if cluster_name else f"Cluster {cluster_id}"
    except Exception as e:
        import traceback
        print(f"Error generating name for cluster {cluster_id}:")
        print(f"Error type: {type(e).__name__}")
        print(f"Error message: {str(e)}")
        print(f"Full traceback:\n{traceback.format_exc()}")
        return f"Cluster {cluster_id}"
#testing
import pandas as pd
embeddings_df = pd.read_pickle('../data/complaint_embeddings_with_clusters.pkl')
embeddings_df.head(2)

,0,1,2,3,4,5,6,7,8,9,...,380,381,382,383,complaint_id,date_received,company,product,consumer_complaint_narrative,agglomerative_cluster
0,-0.370672,-0.297977,0.118031,0.151081,0.284094,-0.118132,0.145074,-0.163338,-0.204628,-0.348756,...,0.212616,-0.059139,0.534717,-0.120117,16179108,2025-09-25,WELLS FARGO & COMPANY,Credit reporting or other personal consumer re...,THESE 3 COMPANIES ARE ON MY CREDIT REPORT AND ...,24
1,-0.111187,-0.089718,0.415655,0.017798,0.136114,-0.181542,0.113605,-0.134354,-0.011274,-0.133676,...,0.269920,-0.069039,0.081852,-0.046670,15305109,2025-08-15,CAPITAL ONE FINANCIAL CORPORATION,Credit reporting or other personal consumer re...,"On XX/XX/scrub>, 2025 I received a text saying...",6


In [2]:

sample_narratives = sample_narratives_by_product_cluster(
    embeddings_df, 
    product='Credit reporting, credit repair services, or other personal consumer reports', 
    cluster_col='agglomerative_cluster',
    n_samples=10
)
cluster_name = create_cluster_name(0, sample_narratives)
print(cluster_name)

c:\Users\whdgu\OneDrive\Desktop\complaints\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Raw response: ChatCompletionOutput(choices=[ChatCompletionOutputComplete(finish_reason='stop', index=0, message=ChatCompletionOutputMessage(role='assistant', content='Product Quality Issues', reasoning=None, tool_call_id=None, tool_calls=None), logprobs=None)], created=1764519443, id='5573fe765cd5ed849b6864e38c48a48a', model='qwen/qwen-2.5-72b-instruct', system_fingerprint='', usage=ChatCompletionOutputUsage(completion_tokens=3, prompt_tokens=27, total_tokens=30, prompt_tokens_details=None, completion_tokens_details=None), object='chat.completion')
Extracted cluster name: Product Quality Issues
Product Quality Issues
